In [1]:
import pandas as pd
import requests
import geopandas as gpd
from rasterstats import zonal_stats
import os, glob
import rasterio
import numpy as np
from rasterio.features import rasterize
from datetime import datetime
from dateutil.relativedelta import relativedelta
from urllib.parse import urlencode
import re

raster_folder = "./data"

In [2]:
import pandas as pd
import glob
import os

# Find all rainfall and temperature CSV files
files = glob.glob('../public/rainfall/*_rainfall.csv') + glob.glob('../public/temperature/*_temperature.csv')

for file_path in files:
    # Load the wide data
    df = pd.read_csv(file_path)
    
    # Identify the first column (e.g., 'division', 'island', etc.)
    id_col = df.columns[0]
    
    # Unpivot (melt) the data from wide to long format
    df_unpivoted = df.melt(id_vars=[id_col], var_name='date', value_name='value')
    
    # Standardize the first column name to 'division'
    df_unpivoted.rename(columns={id_col: 'division'}, inplace=True)
    
    # Save the new format
    output_path = file_path.replace('.csv', '.csv')
    df_unpivoted.to_csv(output_path, index=False)
    print(f"Successfully converted {file_path} to {output_path}")

Successfully converted ../public/rainfall/watershed_rainfall.csv to ../public/rainfall/watershed_rainfall.csv
Successfully converted ../public/rainfall/climate_rainfall.csv to ../public/rainfall/climate_rainfall.csv
Successfully converted ../public/rainfall/island_rainfall.csv to ../public/rainfall/island_rainfall.csv
Successfully converted ../public/rainfall/moku_rainfall.csv to ../public/rainfall/moku_rainfall.csv
Successfully converted ../public/rainfall/ahupuaa_rainfall.csv to ../public/rainfall/ahupuaa_rainfall.csv
Successfully converted ../public/rainfall/statewide_rainfall.csv to ../public/rainfall/statewide_rainfall.csv
Successfully converted ../public/temperature/ahupuaa_temperature.csv to ../public/temperature/ahupuaa_temperature.csv
Successfully converted ../public/temperature/moku_temperature.csv to ../public/temperature/moku_temperature.csv
Successfully converted ../public/temperature/island_temperature.csv to ../public/temperature/island_temperature.csv
Successfully conve

In [3]:
def get_key_from_environment(file_path, key):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    pattern = rf'{key}\s*:\s*[\'"]([^\'"]+)[\'"]'
    match = re.search(pattern, content)
    return match.group(1) if match else None

API_KEY = get_key_from_environment("../src/environments/environment.ts", "apiToken")

In [3]:

DATASETS = {
    "drought": {
        "url": "https://api.hcdp.ikewai.org/raster",
        "params": {
            "datatype": "spi",
            "period": "month",
            "timescale": None  # placeholder
        },
        "scales": [1, 6, 12]
    },
    "rainfall": {
        "url": "https://api.hcdp.ikewai.org/raster",
        "params": {
            "datatype": "rainfall",
            "production": "new",
            "period": "month"
        },
        "scales": [None]
    },
    "temperature": {
        "url": "https://api.hcdp.ikewai.org/raster",
        "params": {
            "datatype": "temperature",
            "aggregation": "mean",
            "period": "month"
        },
        "scales": [None]
    }
}

START_DATE = datetime(2020, 9, 1)
END_DATE = datetime(2025, 8, 1)
RASTER_DIR = "./data"

In [4]:
def fetch_tifs(dataset, scale=None):
    """Download all rasters for given dataset and scale."""
    info = DATASETS[dataset]
    headers = {"Authorization": f"Bearer {API_KEY}"}

    params = info["params"].copy()
    if scale:
        params["timescale"] = f"timescale{scale:03d}"

    date = START_DATE
    while date <= END_DATE:
        params["date"] = date.strftime("%Y-%m")
        query = urlencode(params)
        url = f"{info['url']}?{query}"

        out_path = f"{RASTER_DIR}/" \
        f"{dataset if dataset != 'drought' else ''}" \
        f"{f'spi{scale:03d}' if scale else ''}" \
        f"_{params['date']}.tif"

        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            with open(out_path, "wb") as f:
                f.write(res.content)
            print(f"Saved {out_path}")
        else:
            print(f"{res.status_code} for {url}")

        date += relativedelta(months=1)

In [8]:
def get_averages_rainfall(division, id_col):
    """
    Compute mean rainfall per polygon and export to CSV with unique keys like:
    'Hawaiʻi::Kona-1' or 'Maui::Honuaʻula'
    """

    shp_path = f"../public/shapefiles/{division}.shp"
    gdf = gpd.read_file(shp_path).reset_index(drop=True)

    possible_island_cols = ["island", "ISLAND", "mokupuni", "Mokupuni", "isle"]
    island_col = next((c for c in possible_island_cols if c in gdf.columns), None)

    # Combine same names eg Hilo climate division
    # if division in ["county", "climate"]:
    #   if island_col:
    #       gdf = gdf.dissolve(by=[island_col, id_col], as_index=False)
    #   else:
    #       gdf = gdf.dissolve(by=id_col, as_index=False)

    if island_col:
        gdf["division_full"] = gdf.apply(
            lambda r: f"{r[island_col]}::{r[id_col]}" if pd.notna(r[island_col]) else r[id_col],
            axis=1
        )
    else:
        gdf["division_full"] = gdf[id_col]

    # Collect rainfall rasters
    tifs = sorted(glob.glob(os.path.join(raster_folder, "rainfall_*.tif")))
    if not tifs:
        raise FileNotFoundError("No rainfall rasters found")

    # Base raster metadata
    with rasterio.open(tifs[0]) as src:
        raster_crs = src.crs
        transform = src.transform
        shape = (src.height, src.width)

    if gdf.crs != raster_crs:
        gdf = gdf.to_crs(raster_crs)

    # Rasterize polygons
    shapes = [(geom, idx) for idx, geom in enumerate(gdf.geometry)]
    mask = rasterize(shapes, out_shape=shape, transform=transform, fill=-1, dtype="int32")

    # Extract means per time step
    records = []
    for tif in tifs:
        # rainfall_YYYY-MM.tif
        date = os.path.basename(tif).split("_")[1].replace(".tif", "")
        with rasterio.open(tif) as src:
            arr = src.read(1, masked=True)

        for idx, div in enumerate(gdf["division_full"]):
            poly_mask = mask == idx
            vals = arr[poly_mask]
            if np.ma.is_masked(vals):
                vals = vals.compressed()
            mean_val = np.nan if vals.size == 0 else np.nanmean(vals)
            records.append({"division": div, "date": date, "mean_rain": mean_val/25.4})

    # Build tidy table
    df = (
        pd.DataFrame(records)
        .groupby(["division", "date"])["mean_rain"]
        .mean()
        .reset_index()
        .pivot(index="division", columns="date", values="mean_rain")
    )

    df = df.reindex(sorted(df.columns), axis=1)

    out_csv = f"../public/rainfall/{division}_rainfall.csv"
    df.to_csv(out_csv)
    print(f"Saved {out_csv} ({len(df)} rows)")

In [10]:
#moku, moku
#climate, name
#ahupuaa, ahupuaa
#island, name
get_averages_rainfall("island", "name")

Saved ../public/rainfall/island_rainfall.csv (7 rows)


In [ ]:
# fetch_tifs("rainfall")

Saved ./data/rainfall_2020-09.tif
Saved ./data/rainfall_2020-10.tif
Saved ./data/rainfall_2020-11.tif
Saved ./data/rainfall_2020-12.tif
Saved ./data/rainfall_2021-01.tif
Saved ./data/rainfall_2021-02.tif
Saved ./data/rainfall_2021-03.tif
Saved ./data/rainfall_2021-04.tif
Saved ./data/rainfall_2021-05.tif
Saved ./data/rainfall_2021-06.tif
Saved ./data/rainfall_2021-07.tif
Saved ./data/rainfall_2021-08.tif
Saved ./data/rainfall_2021-09.tif
Saved ./data/rainfall_2021-10.tif
Saved ./data/rainfall_2021-11.tif
Saved ./data/rainfall_2021-12.tif
Saved ./data/rainfall_2022-01.tif
Saved ./data/rainfall_2022-02.tif
Saved ./data/rainfall_2022-03.tif
Saved ./data/rainfall_2022-04.tif
Saved ./data/rainfall_2022-05.tif
Saved ./data/rainfall_2022-06.tif
Saved ./data/rainfall_2022-07.tif
Saved ./data/rainfall_2022-08.tif
Saved ./data/rainfall_2022-09.tif
Saved ./data/rainfall_2022-10.tif
Saved ./data/rainfall_2022-11.tif
Saved ./data/rainfall_2022-12.tif
Saved ./data/rainfall_2023-01.tif
Saved ./data/r

In [3]:
#statewide
shapefile = "../public/shapefiles/Coastline.shp"
raster_folder = "./data"

# Load shapefile and dissolve to one feature (statewide)
gdf = gpd.read_file(shapefile)
gdf_statewide = gdf.dissolve()  # merges all polygons into one

records = []

for tif in sorted(glob.glob(os.path.join(raster_folder, f"rainfall_*.tif"))):
    # Extract date from filename (spi001_2024-09.tif → 2024-09)
    date = os.path.basename(tif).split("_")[1].replace(".tif", "")
    with rasterio.open(tif) as src:
        nodata = src.nodata
    # Compute zonal stats for the dissolved geometry
    stats = zonal_stats(
        vectors=gdf_statewide,
        raster=tif,
        stats=["mean"],
        geojson_out=False,
        nodata=nodata
    )
    
    records.append({
        "date": date,
        "value": stats[0]["mean"]/25.4
    })


# Pivot to wide format: one row (statewide), months as columns
df = pd.DataFrame(records).set_index("date").T
df.insert(0, "state", "statewide")  # add proper state column
df.to_csv(f"../public/rainfall/statewide_rf.csv", index=False)

print(df)

date       state   2020-09   2020-10   2020-11   2020-12   2021-01   2021-02  \
value  statewide  4.192759  3.213976  8.686313  3.823757  9.113911  4.916769   

date    2021-03   2021-04   2021-05  ...   2024-11   2024-12   2025-01  \
value  13.99668  4.477529  3.275408  ...  6.510566  1.222673  6.122403   

date    2025-02   2025-03   2025-04   2025-05   2025-06  2025-07   2025-08  
value  1.234921  4.272403  4.170727  3.213287  3.397662  3.09688  1.995946  

[1 rows x 61 columns]


In [30]:
#Remove files
folder = "./data"

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    if os.path.isfile(file_path):  # ensures it's a file
        os.remove(file_path)